In [3]:
pip install requests stackstac pystac_client planetary_computer torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.2/111.2 MB 7.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 8.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 7.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.1.0
    Uninstalling setuptools-75.1.0:
      Successfully uninstalled setuptools-75.1.0
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import re
import warnings
import xml.etree.ElementTree as ET
 
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import stackstac
import pystac_client
import planetary_computer
import torch
import torch.nn as nn
from rasterio.windows import from_bounds
from rasterio.features import rasterize
from sklearn.metrics import f1_score, jaccard_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

In [9]:
BUCKET = "https://usgs-landcover.s3.us-west-2.amazonaws.com"
NS = "{http://s3.amazonaws.com/doc/2006-03-01/}"
TIGER = "https://www2.census.gov/geo/tiger/TIGER2024/PLACE/tl_2024_%s_place.zip"
STAC = "https://planetarycomputer.microsoft.com/api/stac/v1"
 
CITIES = {"Philadelphia": ("42", "4260000"),
          "Detroit": ("26", "2622000"),
          "Atlanta": ("13", "1304000")}
HOME = "Philadelphia"
T0, T1 = 2015, 2025
BANDS = ["blue", "green", "red", "nir08", "swir16", "swir22", "lwir11"]
DEVELOPED = (21, 22, 23, 24)
WATER = (11, 12)
WETLAND = (90, 95)
IMP_DROP = 15
PIXEL_M = 30
PATCH = 128
STRIDE = 64
BLOCK_PX = 256
EPOCHS = 40
BATCH = 16
SEED = 650

In [10]:
CLASSES = ["No change", "Sprawl", "Decay"]
INK = "#12100E"
MUTE = "#6E6A65"
LAND = "#E6E2DA"
SPRAWL_C = "#F2914F"
DECAY_C = "#8E1B2E"

In [19]:
import tempfile
os.chdir("/Users/cyberhbliu/Desktop/PERSONAL/2026portfolio/urban_decay_and_sprawl")
os.makedirs("figures", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
 
plt.rcParams.update({"figure.facecolor": "#FFFFFF", "axes.facecolor": "#FFFFFF",
                     "savefig.facecolor": "#FFFFFF", "font.family": "DejaVu Sans",
                     "text.color": INK, "axes.edgecolor": "#D8D4CC",
                     "xtick.color": MUTE, "ytick.color": MUTE, "figure.dpi": 150})


# 1. fetching NLCD labels over WMS

In [23]:
WMS = {"LndCov": ("https://dmsdata.cr.usgs.gov/geoserver/mrlc_Land-Cover-Native_conus_year_data/wms",
                  "Land-Cover-Native_conus_year_data"),
       "FctImp": ("https://dmsdata.cr.usgs.gov/geoserver/mrlc_Fractional-Impervious-Surface-Native_conus_year_data/wms",
                  "Fractional-Impervious-Surface-Native_conus_year_data")}
 
NLCD_RGB = {(70, 107, 159): 11, (209, 222, 248): 12, (222, 197, 197): 21,
            (217, 146, 130): 22, (235, 0, 0): 23, (171, 0, 0): 24,
            (179, 172, 159): 31, (104, 171, 95): 41, (28, 95, 44): 42,
            (181, 197, 143): 43, (204, 184, 121): 52, (223, 223, 194): 71,
            (220, 217, 57): 81, (171, 108, 40): 82, (184, 217, 235): 90,
            (108, 159, 184): 95}
 
TILE = 1800
GRID = {}
RASTER = {}

In [ ]:
for city, (state, place) in CITIES.items():
    shape = gpd.read_file(TIGER % state)
    shape = shape[shape["GEOID"] == place].to_crs("EPSG:5070")
    minx, miny, maxx, maxy = shape.total_bounds
    minx = np.floor(minx / PIXEL_M) * PIXEL_M
    maxy = np.ceil(maxy / PIXEL_M) * PIXEL_M
    width = int(np.ceil((maxx - minx) / PIXEL_M))
    height = int(np.ceil((maxy - miny) / PIXEL_M))
    maxx, miny = minx + width * PIXEL_M, maxy - height * PIXEL_M
    GRID[city] = {"shape": shape, "width": width, "height": height,
                  "bounds": (minx, miny, maxx, maxy),
                  "transform": rasterio.transform.from_origin(minx, maxy, PIXEL_M, PIXEL_M)}
 
    for product in ("LndCov", "FctImp"):
        endpoint, layer = WMS[product]
        for year in (T0, T1):
            canvas = None
            for r0 in range(0, height, TILE):
                for c0 in range(0, width, TILE):
                    h = min(TILE, height - r0)
                    w = min(TILE, width - c0)
                    bx0 = minx + c0 * PIXEL_M
                    by1 = maxy - r0 * PIXEL_M
                    resp = requests.get(endpoint, params={
                        "service": "WMS", "version": "1.1.1", "request": "GetMap",
                        "layers": layer, "styles": "", "srs": "EPSG:5070",
                        "bbox": "%f,%f,%f,%f" % (bx0, by1 - h * PIXEL_M,
                                                 bx0 + w * PIXEL_M, by1),
                        "width": w, "height": h, "format": "image/geotiff",
                        "transparent": "false",
                        "time": "%d-01-01T00:00:00.000Z" % year,
                    }, timeout=300)
                    resp.raise_for_status()
                    if b"ServiceException" in resp.content[:2000]:
                        raise SystemExit(resp.content[:600].decode("utf-8", "ignore"))
                    with rasterio.MemoryFile(resp.content) as mem:
                        with mem.open() as src:
                            block = src.read()
                    if canvas is None:
                        canvas = np.zeros((block.shape[0], height, width), block.dtype)
                    canvas[:, r0:r0 + h, c0:c0 + w] = block[:, :h, :w]
 
            if canvas.shape[0] == 1:
                arr = canvas[0]
            elif product == "LndCov":
                arr = np.zeros((height, width), np.uint8)
                rgb = canvas[:3].transpose(1, 2, 0)
                for colour, code in NLCD_RGB.items():
                    arr[np.all(rgb == np.array(colour, canvas.dtype), axis=-1)] = code
                hit = (arr > 0).mean()
                print("      %-13s %s %d  RGB decode matched %.1f%%" % (city, product, year, 100 * hit))
                if hit < 0.9:
                    raise SystemExit("RGB decode failed, the WMS style is not the standard NLCD legend")
            else:
                arr = None
            RASTER[(city, product, year)] = arr
            if arr is not None and canvas.shape[0] == 1:
                print("      %-13s %s %d  %d x %d  values %s"
                      % (city, product, year, height, width,
                         np.unique(arr)[:8]))
 
USE_IMPERVIOUS = all(RASTER[(c, "FctImp", y)] is not None
                     for c in CITIES for y in (T0, T1))
print("      decay signal: %s" % ("impervious decline" if USE_IMPERVIOUS
                                  else "land cover intensity downgrade"))

# 2. building imagery and labels

In [ ]:
catalog = pystac_client.Client.open(STAC, modifier=planetary_computer.sign_inplace)
INTENSITY = {21: 1, 22: 2, 23: 3, 24: 4}
DATA = {}
 
for city in CITIES:
    g = GRID[city]
    shape, transform = g["shape"], g["transform"]
    height, width = g["height"], g["width"]
    minx, miny, maxx, maxy = g["bounds"]
 
    cov0 = RASTER[(city, "LndCov", T0)]
    cov1 = RASTER[(city, "LndCov", T1)]
    inside = rasterize([(x, 1) for x in shape.geometry], out_shape=(height, width),
                       transform=transform, fill=0, dtype="uint8").astype(bool)
 
    dev0, dev1 = np.isin(cov0, DEVELOPED), np.isin(cov1, DEVELOPED)
    blocked = np.isin(cov0, WATER) | np.isin(cov0, WETLAND)
 
    if USE_IMPERVIOUS:
        imp0 = RASTER[(city, "FctImp", T0)].astype(np.int16)
        imp1 = RASTER[(city, "FctImp", T1)].astype(np.int16)
        decline = dev0 & ((imp0 - imp1) >= IMP_DROP)
    else:
        rank0 = np.zeros((height, width), np.int8)
        rank1 = np.zeros((height, width), np.int8)
        for code, level in INTENSITY.items():
            rank0[cov0 == code] = level
            rank1[cov1 == code] = level
        decline = dev0 & (rank1 < rank0)
 
    label = np.zeros((height, width), np.int64)
    label[(~dev0) & (~blocked) & dev1] = 1
    label[decline] = 2
    label[~inside] = 0
 
    wgs = shape.to_crs("EPSG:4326").total_bounds
    scenes = []
    for year in (T0, T1):
        items = catalog.search(
            collections=["landsat-c2-l2"],
            bbox=list(wgs),
            datetime="%d-05-01/%d-09-30" % (year, year),
            query={"eo:cloud_cover": {"lt": 30},
                   "platform": {"in": ["landsat-8", "landsat-9"]}},
        ).item_collection()
        if len(items) == 0:
            raise SystemExit("no Landsat scenes for %s %d" % (city, year))
 
        cube = stackstac.stack(items, assets=BANDS + ["qa_pixel"], epsg=5070,
                               resolution=PIXEL_M, bounds=(minx, miny, maxx, maxy),
                               chunksize=1024, dtype="float32", fill_value=np.nan)
        qa = cube.sel(band="qa_pixel").astype("uint16")
        clear = ((qa & (1 << 1)) == 0) & ((qa & (1 << 3)) == 0) & ((qa & (1 << 4)) == 0)
        composite = cube.sel(band=BANDS).where(clear).median("time").compute().values
 
        optical = composite[:6] * 0.0000275 - 0.2
        thermal = (composite[6] * 0.00341802 + 149.0 - 273.15) / 50.0
        nd = lambda a, b: (a - b) / (a + b + 1e-6)
        scenes.append(np.concatenate([optical, thermal[None],
                                      nd(optical[3], optical[2])[None],
                                      nd(optical[4], optical[3])[None]]).astype(np.float32))
        print("      %-13s %d  %d scenes" % (city, year, len(items)))
 
    image = np.nan_to_num(np.concatenate(scenes), nan=0.0)
    if image.shape[1:] != (height, width):
        pad = np.zeros((image.shape[0], height, width), np.float32)
        h = min(height, image.shape[1])
        w = min(width, image.shape[2])
        pad[:, :h, :w] = image[:, :h, :w]
        image = pad
 
    rows, cols = np.mgrid[0:height, 0:width]
    checker = ((rows // BLOCK_PX) + (cols // BLOCK_PX)) % 2
 
    DATA[city] = {"image": image, "label": label, "inside": inside,
                  "checker": checker, "shape": shape, "transform": transform,
                  "extent": (minx, maxx, miny, maxy)}
    print("      %-13s %d x %d  sprawl %d  decay %d"
          % (city, height, width, (label == 1).sum(), (label == 2).sum()))